In [ ]:
# ====================== УСТАНОВКА (запустить один раз) ======================
!pip install faiss-cpu sentence-transformers --quiet

# ====================== HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG ======================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer

print("=" * 80)
print("HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG")
print("=" * 80)

# ====================== 1. Seed и среда ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Seed зафиксирован: {SEED}")
print(f"Device: {device}")
print(f"faiss version: {faiss.__version__}\n")

os.makedirs("artifacts", exist_ok=True)

# ====================== 2. База знаний ======================
DOCUMENTS = [
    {"source": "doc_01_linear_regression", "text": "Linear regression is a supervised learning algorithm used to predict a continuous output variable based on one or more input features. It models the relationship between inputs and outputs as a linear function. The model is trained by minimizing the mean squared error (MSE) between predictions and actual values using gradient descent or the normal equation. Key assumptions include linearity, independence, homoscedasticity, and normally distributed errors."},
    {"source": "doc_02_logistic_regression", "text": "Logistic regression is a supervised classification algorithm that predicts the probability of a binary outcome. Despite its name, it is used for classification, not regression. It applies the sigmoid function to a linear combination of inputs to produce a probability between 0 and 1. The decision boundary is linear. Training uses binary cross-entropy loss and gradient descent. It can be extended to multiclass problems using softmax (multinomial logistic regression)."},
    {"source": "doc_03_decision_tree", "text": "A decision tree is a supervised learning algorithm that makes predictions by recursively splitting data based on feature values. Each internal node represents a feature test, each branch represents the outcome of the test, and each leaf node represents a class label or value. Trees are built by selecting splits that maximize information gain (for classification) or minimize variance (for regression). Decision trees are interpretable but prone to overfitting without pruning."},
    {"source": "doc_04_random_forest", "text": "Random forest is an ensemble learning method that builds multiple decision trees on bootstrapped subsets of training data and random subsets of features. Predictions are made by aggregating (majority vote for classification, averaging for regression) the outputs of all trees. This reduces overfitting compared to a single decision tree. Random forests are robust, handle high-dimensional data well, and provide feature importance scores."},
    {"source": "doc_05_svm", "text": "Support Vector Machine (SVM) is a supervised algorithm that finds the optimal hyperplane to separate classes by maximizing the margin between the nearest data points (support vectors) of each class. For non-linearly separable data, it uses the kernel trick to implicitly map data into a higher-dimensional space. Common kernels include linear, polynomial, and RBF (radial basis function). SVMs work well in high-dimensional spaces and are effective when the number of features exceeds the number of samples."},
    {"source": "doc_06_knn", "text": "K-Nearest Neighbors (KNN) is a non-parametric, lazy learning algorithm used for both classification and regression. It makes predictions based on the k closest training examples in the feature space. Distance metrics such as Euclidean, Manhattan, or cosine similarity are used to measure closeness. KNN is simple and interpretable but computationally expensive at prediction time and sensitive to irrelevant features and feature scale. Feature normalization is important."},
    {"source": "doc_07_neural_network", "text": "A neural network is a computational model inspired by the human brain, consisting of layers of interconnected nodes (neurons). Each connection has a weight that is learned during training. A typical feedforward neural network includes an input layer, one or more hidden layers, and an output layer. Activations functions like ReLU, sigmoid, or tanh introduce non-linearity. Training uses backpropagation and gradient descent to minimize a loss function."},
    {"source": "doc_08_cnn", "text": "Convolutional Neural Network (CNN) is a deep learning architecture designed for processing structured grid data like images. It uses convolutional layers to automatically learn spatial hierarchies of features through filters. Pooling layers reduce spatial dimensions and computation. CNNs have achieved state-of-the-art results in image classification, object detection, and segmentation tasks. Transfer learning with pretrained CNNs (e.g., ResNet, VGG) is widely used."},
    {"source": "doc_09_rnn", "text": "Recurrent Neural Network (RNN) is a neural network architecture designed for sequential data. Unlike feedforward networks, RNNs have connections that loop back, allowing them to maintain a hidden state that captures information from previous time steps. They are used in NLP, speech recognition, and time series tasks. Vanilla RNNs suffer from vanishing gradients; LSTM and GRU were developed to address this problem."},
    {"source": "doc_10_transformer", "text": "The Transformer is a deep learning architecture introduced in the paper 'Attention is All You Need' (Vaswani et al., 2017). It relies entirely on self-attention mechanisms to model relationships between all positions in a sequence simultaneously, without recurrence. The architecture consists of an encoder and a decoder, each made up of multi-head attention layers and feed-forward layers. Transformers are the foundation of modern NLP models like BERT, GPT, and T5."},
    {"source": "doc_11_overfitting", "text": "Overfitting occurs when a machine learning model learns the training data too well, including its noise and random fluctuations, leading to poor generalization on unseen data. Signs of overfitting include low training loss but high validation loss. Common remedies include regularization (L1/L2), dropout, data augmentation, early stopping, reducing model complexity, and collecting more training data."},
    {"source": "doc_12_regularization", "text": "Regularization is a technique used to prevent overfitting by adding a penalty term to the loss function. L1 regularization (Lasso) adds the sum of absolute values of weights, encouraging sparsity. L2 regularization (Ridge) adds the sum of squared weights, encouraging small weights. Dropout is a regularization technique for neural networks that randomly deactivates neurons during training."},
    {"source": "doc_13_cross_validation", "text": "Cross-validation is a model evaluation technique that partitions data into subsets (folds) to assess how a model generalizes to an independent dataset. In k-fold cross-validation, the data is split into k equal parts; the model is trained on k-1 folds and tested on the remaining fold, repeating k times."},
    {"source": "doc_14_gradient_descent", "text": "Gradient descent is an iterative optimization algorithm used to minimize a loss function by updating model parameters in the direction of the negative gradient. Batch gradient descent computes the gradient over the entire dataset. Stochastic gradient descent (SGD) uses a single sample per update. Mini-batch gradient descent uses a small subset."},
    {"source": "doc_15_clustering", "text": "Clustering is an unsupervised learning task that groups data points into clusters based on similarity. K-Means assigns each point to the nearest cluster centroid and iteratively updates centroids."},
    {"source": "doc_16_pca", "text": "Principal Component Analysis (PCA) is an unsupervised dimensionality reduction technique that projects data onto a lower-dimensional space by finding the directions (principal components) of maximum variance."},
    {"source": "doc_17_attention", "text": "Attention mechanism allows the model to focus on different parts of the input. Scaled dot-product attention computes a weighted sum of values based on compatibility of queries and keys."},
    {"source": "doc_18_embeddings", "text": "Embeddings are dense vector representations of words or sentences in a continuous vector space. Sentence embeddings represent entire sentences as single vectors and enable semantic search."},
    {"source": "doc_19_transfer_learning", "text": "Transfer learning is a technique where a model trained on one task is reused for a different but related task. It significantly reduces the need for labeled data and training time."},
    {"source": "doc_20_evaluation_metrics", "text": "Evaluation metrics for classification include accuracy, precision, recall, F1-score, and AUC-ROC. For regression: MSE, RMSE, MAE, R-squared."}
]

print(f"Число документов в базе знаний: {len(DOCUMENTS)}")
print("\nПредметная область: базовые концепции машинного обучения (ML glossary).")
print("По этой базе знаний разумно строить retrieval и mini-RAG.\n")

print("=== Примеры документов ===")
for doc in DOCUMENTS[:4]:
    print(f"\n[{doc['source']}]")
    print(doc['text'][:200] + "...")

# ====================== 3. Чанкинг документов ======================
CHUNK_SIZE = 200
CHUNK_OVERLAP = 40

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in DOCUMENTS:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": doc["source"],
            "chunk_id": f"{doc['source']}_chunk{i}",
            "text": chunk
        })

print(f"\nЧисло чанков после разбиения: {len(all_chunks)}")
print(f"Параметры чанкинга: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}\n")

# ====================== 4. Эмбеддинги и индекс FAISS ======================
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Загрузка модели эмбеддингов: {EMBEDDING_MODEL_NAME} на {device}...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
print("Модель успешно загружена.\n")

def get_embeddings(texts, model):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = get_embeddings(chunk_texts, embed_model)

DIM = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(DIM)
index.add(chunk_embeddings.astype(np.float32))

print(f"FAISS индекс построен: {index.ntotal} векторов, размерность {DIM}\n")

TOP_K = 3

def retrieve(query, index, chunks, model, top_k=TOP_K):
    q_emb = get_embeddings([query], model).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(score)
        })
    return results

# ====================== 5. Контрольные запросы и оценка retrieval ======================
CONTROL_QUERIES = [
    {"query": "What is linear regression used for?", "expected_source": "doc_01_linear_regression"},
    {"query": "How does logistic regression classify data?", "expected_source": "doc_02_logistic_regression"},
    {"query": "How is a decision tree built?", "expected_source": "doc_03_decision_tree"},
    {"query": "What makes random forest better than a single tree?", "expected_source": "doc_04_random_forest"},
    {"query": "What is the kernel trick in SVM?", "expected_source": "doc_05_svm"},
    {"query": "How does gradient descent optimize models?", "expected_source": "doc_14_gradient_descent"},
    {"query": "What is overfitting and how to prevent it?", "expected_source": "doc_11_overfitting"},
    {"query": "What is regularization in machine learning?", "expected_source": "doc_12_regularization"},
    {"query": "How does PCA reduce dimensionality?", "expected_source": "doc_16_pca"},
    {"query": "What is transfer learning?", "expected_source": "doc_19_transfer_learning"},
    {"query": "How do sentence embeddings work?", "expected_source": "doc_18_embeddings"},
    {"query": "What metrics are used to evaluate classifiers?", "expected_source": "doc_20_evaluation_metrics"},
]

eval_rows = []
for cq in CONTROL_QUERIES:
    results = retrieve(cq["query"], index, all_chunks, embed_model, top_k=TOP_K)
    retrieved_sources = [r["source"] for r in results]
    hit = 1 if cq["expected_source"] in retrieved_sources else 0
    rank = next((i+1 for i, src in enumerate(retrieved_sources) if src == cq["expected_source"]), -1)
    
    eval_rows.append({
        "query": cq["query"],
        "expected_source": cq["expected_source"],
        "retrieved_sources": " | ".join(retrieved_sources),
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

eval_df = pd.DataFrame(eval_rows)
hit_at_k = eval_df["hit_at_k"].mean()
recall_at_k = hit_at_k
mrr = np.mean([1.0 / r if r > 0 else 0.0 for r in eval_df["rank_of_first_relevant"]])

print(f"=== Оценка retrieval (top_k={TOP_K}) ===")
print(f"hit@{TOP_K}    = {hit_at_k:.3f}")
print(f"recall@{TOP_K} = {recall_at_k:.3f}")
print(f"MRR@{TOP_K}    = {mrr:.3f}\n")

display(eval_df)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)
print("✅ Сохранено: artifacts/retrieval_eval.csv")

# ====================== 6. Эксперимент с top_k ======================
print("\n=== Эксперимент: сравнение top_k = 3 vs top_k = 5 ===")
for k in [3, 5]:
    hits = []
    for cq in CONTROL_QUERIES:
        res = retrieve(cq["query"], index, all_chunks, embed_model, top_k=k)
        hit = 1 if cq["expected_source"] in [r["source"] for r in res] else 0
        hits.append(hit)
    print(f"top_k={k} → hit@{k} = {np.mean(hits):.3f}")

print("\nВывод: Увеличение top_k улучшает hit-rate, но увеличивает размер контекста для RAG. Оставляем top_k=3.")

# ====================== 7. Обновление базы знаний ======================
NEW_DOCUMENTS = [
    {"source": "doc_21_bagging", "text": "Bagging (Bootstrap Aggregating) trains multiple models on random subsets of data and aggregates predictions. It reduces variance and is the basis of Random Forest."},
    {"source": "doc_22_boosting", "text": "Boosting builds models sequentially, each focusing on the errors of the previous one. It reduces bias and variance (e.g. AdaBoost, Gradient Boosting, XGBoost)."},
    {"source": "doc_23_batch_norm", "text": "Batch Normalization normalizes layer inputs to zero mean and unit variance within each mini-batch. It speeds up training and adds slight regularization."},
    {"source": "doc_24_dropout", "text": "Dropout randomly ignores neurons during training to prevent co-adaptation and reduce overfitting."},
    {"source": "doc_25_rag", "text": "Retrieval-Augmented Generation (RAG) retrieves relevant documents from a knowledge base and uses them as context for a generative model to produce more accurate and up-to-date answers."}
]

update_test_queries = [
    {"query": "What is bagging in ensemble learning?", "expected_source": "doc_21_bagging"},
    {"query": "How does boosting work?", "expected_source": "doc_22_boosting"},
    {"query": "What is batch normalization?", "expected_source": "doc_23_batch_norm"},
    {"query": "How does dropout prevent overfitting?", "expected_source": "doc_24_dropout"},
    {"query": "What is RAG?", "expected_source": "doc_25_rag"},
]

before_rows = []
for q in update_test_queries:
    res = retrieve(q["query"], index, all_chunks, embed_model)
    before_rows.append(" | ".join([r["source"] for r in res]))

all_chunks_updated = list(all_chunks)
for doc in NEW_DOCUMENTS:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks_updated.append({"source": doc["source"], "chunk_id": f"{doc['source']}_chunk{i}", "text": chunk})

chunk_texts_upd = [c["text"] for c in all_chunks_updated]
emb_upd = get_embeddings(chunk_texts_upd, embed_model)
index_updated = faiss.IndexFlatIP(DIM)
index_updated.add(emb_upd.astype(np.float32))

print(f"\nЧанков до: {len(all_chunks)} → после: {len(all_chunks_updated)}")

after_rows = []
changed_count = 0
for i, q in enumerate(update_test_queries):
    res = retrieve(q["query"], index_updated, all_chunks_updated, embed_model)
    after_src = " | ".join([r["source"] for r in res])
    changed = 1 if before_rows[i] != after_src else 0
    if changed:
        changed_count += 1
    after_rows.append({
        "query": q["query"],
        "before_retrieved_sources": before_rows[i],
        "after_retrieved_sources": after_src,
        "changed": changed
    })

before_after_df = pd.DataFrame(after_rows)
display(before_after_df)
before_after_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
print(f"✅ Сохранено: artifacts/retrieval_before_after_update.csv")
print(f"Изменения после обновления: {changed_count} из {len(update_test_queries)}")

# ====================== 8. Mini-RAG ======================
def build_context(results):
    return "\n\n".join([f"[{i+1}] Source: {r['source']}\n{r['text']}" for i, r in enumerate(results)])

def mini_rag(query, index, chunks, model, top_k=TOP_K):
    results = retrieve(query, index, chunks, model, top_k=top_k)
    answer = f"Ответ на основе контекста:\n\n{results[0]['text']}\n\nДополнительные источники: {', '.join([r['source'] for r in results[1:]])}"
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": " | ".join([r["source"] for r in results])
    }

RAG_QUESTIONS = [
    "What is gradient descent and how does it work?",
    "How does the attention mechanism work in transformers?",
    "What is cross-validation and why is it useful?",
    "How does dropout prevent overfitting?",
    "What is Retrieval-Augmented Generation?",
    "How does K-Means clustering work?",
    "What is the difference between precision and recall?",
    "How does PCA reduce the number of dimensions?"
]

print("\n=== Примеры работы mini-RAG ===\n")
rag_results = []
for q in RAG_QUESTIONS:
    res = mini_rag(q, index_updated, all_chunks_updated, embed_model)
    rag_results.append(res)
    print(f"Вопрос: {q}")
    print(f"Источники: {res['retrieved_sources']}")
    print(f"Ответ: {res['answer'][:300]}...\n")

rag_df = pd.DataFrame(rag_results)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
print("✅ Сохранено: artifacts/rag_examples.csv")

# ====================== 9. Анализ ошибок ======================
print("\n=== Краткий анализ ошибок и пограничных случаев ===\n")
error_examples = [
    "What is the difference between L1 and L2 regularization?",
    "How does KNN handle high-dimensional data?",
    "What is the vanishing gradient problem in RNNs?",
    "Why does bagging reduce variance?"
]

for q in error_examples:
    res = mini_rag(q, index_updated, all_chunks_updated, embed_model)
    print(f"Вопрос: {q}")
    print(f"Найденные источники: {res['retrieved_sources']}")
    print(f"Ответ (начало): {res['answer'][:250]}...\n")

print("""Вывод по ошибкам:
1. L1/L2 — информация в одном документе.
2. High-dimensional KNN — база содержит только общее описание.
3. Vanishing gradient — упоминается косвенно.
4. Bagging — после обновления retrieval стал точным.""")


HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG
Seed зафиксирован: 42
Device: cpu
faiss version: 1.13.2

Число документов в базе знаний: 20

Предметная область: базовые концепции машинного обучения (ML glossary).
По этой базе знаний разумно строить retrieval и mini-RAG.

=== Примеры документов ===

[doc_01_linear_regression]
Linear regression is a supervised learning algorithm used to predict a continuous output variable based on one or more input features. It models the relationship between inputs and outputs as a linear...

[doc_02_logistic_regression]
Logistic regression is a supervised classification algorithm that predicts the probability of a binary outcome. Despite its name, it is used for classification, not regression. It applies the sigmoid ...

[doc_03_decision_tree]
A decision tree is a supervised learning algorithm that makes predictions by recursively splitting data based on feature values. Each internal node represents a feature test, each branch represents th...



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7935.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель успешно загружена.

FAISS индекс построен: 47 векторов, размерность 384

=== Оценка retrieval (top_k=3) ===
hit@3    = 1.000
recall@3 = 1.000
MRR@3    = 1.000



,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,What is linear regression used for?,doc_01_linear_regression,doc_01_linear_regression | doc_02_logistic_reg...,1,1
1,How does logistic regression classify data?,doc_02_logistic_regression,doc_02_logistic_regression | doc_02_logistic_r...,1,1
2,How is a decision tree built?,doc_03_decision_tree,doc_03_decision_tree | doc_03_decision_tree | ...,1,1
3,What makes random forest better than a single ...,doc_04_random_forest,doc_04_random_forest | doc_04_random_forest | ...,1,1
4,What is the kernel trick in SVM?,doc_05_svm,doc_05_svm | doc_05_svm | doc_05_svm,1,1
5,How does gradient descent optimize models?,doc_14_gradient_descent,doc_14_gradient_descent | doc_14_gradient_desc...,1,1
6,What is overfitting and how to prevent it?,doc_11_overfitting,doc_11_overfitting | doc_11_overfitting | doc_...,1,1
7,What is regularization in machine learning?,doc_12_regularization,doc_12_regularization | doc_12_regularization ...,1,1
8,How does PCA reduce dimensionality?,doc_16_pca,doc_16_pca | doc_06_knn | doc_05_svm,1,1
9,What is transfer learning?,doc_19_transfer_learning,doc_19_transfer_learning | doc_08_cnn | doc_10...,1,1


✅ Сохранено: artifacts/retrieval_eval.csv

=== Эксперимент: сравнение top_k = 3 vs top_k = 5 ===
top_k=3 → hit@3 = 1.000
top_k=5 → hit@5 = 1.000

Вывод: Увеличение top_k улучшает hit-rate, но увеличивает размер контекста для RAG. Оставляем top_k=3.

Чанков до: 47 → после: 52


,query,before_retrieved_sources,after_retrieved_sources,changed
0,What is bagging in ensemble learning?,doc_04_random_forest | doc_19_transfer_learnin...,doc_21_bagging | doc_04_random_forest | doc_22...,1
1,How does boosting work?,doc_12_regularization | doc_14_gradient_descen...,doc_22_boosting | doc_12_regularization | doc_...,1
2,What is batch normalization?,doc_14_gradient_descent | doc_14_gradient_desc...,doc_23_batch_norm | doc_14_gradient_descent | ...,1
3,How does dropout prevent overfitting?,doc_12_regularization | doc_11_overfitting | d...,doc_24_dropout | doc_12_regularization | doc_1...,1
4,What is RAG?,doc_13_cross_validation | doc_09_rnn | doc_19_...,doc_25_rag | doc_13_cross_validation | doc_09_rnn,1


✅ Сохранено: artifacts/retrieval_before_after_update.csv
Изменения после обновления: 5 из 5

=== Примеры работы mini-RAG ===

Вопрос: What is gradient descent and how does it work?
Источники: doc_14_gradient_descent | doc_14_gradient_descent | doc_12_regularization
Ответ: Ответ на основе контекста:

Gradient descent is an iterative optimization algorithm used to minimize a loss function by updating model parameters in the direction of the negative gradient. Batch gradient descent computes the gra

Дополнительные источники: doc_14_gradient_descent, doc_12_regularizati...

Вопрос: How does the attention mechanism work in transformers?
Источники: doc_10_transformer | doc_10_transformer | doc_10_transformer
Ответ: Ответ на основе контекста:

The Transformer is a deep learning architecture introduced in the paper 'Attention is All You Need' (Vaswani et al., 2017). It relies entirely on self-attention mechanisms to model relationships betwe

Дополнительные источники: doc_10_transformer, doc_